# 📘 Module 2.2 — CNN Architectures: From LeNet to ResNet

**Goal:** Understand the evolution of CNN architectures and why each innovation matters.

## Architecture Timeline
```
1998: LeNet-5      → First practical CNN (handwriting)
2012: AlexNet      → Deep learning revolution (ImageNet)
2014: VGGNet       → Deeper is better (3x3 kernels)
2014: GoogLeNet    → Multi-scale processing (Inception)
2015: ResNet       → Skip connections solve vanishing gradients
2017: MobileNet    → Efficient CNNs for mobile/edge (ADAS!)
2019: EfficientNet → Optimal scaling of depth/width/resolution
```

---

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np

## 1. LeNet-5 (1998) — The Pioneer

Created by Yann LeCun for handwritten digit recognition. The first CNN to be used in production (postal code reading).

```
Input(1,32,32) → Conv(6) → Pool → Conv(16) → Pool → FC(120) → FC(84) → Output(10)
```

In [2]:
class LeNet5(nn.Module):
    """LeNet-5: The original CNN architecture (1998)."""
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),     # 32→28
            nn.Tanh(),                           # Original used tanh
            nn.AvgPool2d(2, 2),                  # 28→14
            nn.Conv2d(6, 16, kernel_size=5),     # 14→10
            nn.Tanh(),
            nn.AvgPool2d(2, 2),                  # 10→5
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120),
            nn.Tanh(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

lenet = LeNet5()
print(f"LeNet-5 parameters: {sum(p.numel() for p in lenet.parameters()):,}")
x = torch.randn(1, 1, 32, 32)
print(f"Input: {x.shape} → Output: {lenet(x).shape}")

LeNet-5 parameters: 61,706
Input: torch.Size([1, 1, 32, 32]) → Output: torch.Size([1, 10])


## 2. VGG (2014) — Deeper with 3×3 Kernels

**Key Insight:** Two 3×3 convolutions have the same receptive field as one 5×5, but with fewer parameters and more non-linearity.

```
VGG-16: 13 conv layers + 3 FC layers = 138M parameters
Pattern: Conv-Conv-Pool (repeated 5 times)
```

In [3]:
class VGGBlock(nn.Module):
    """A VGG-style block: multiple conv layers followed by max pool."""
    def __init__(self, in_channels, out_channels, num_convs):
        super().__init__()
        layers = []
        for i in range(num_convs):
            layers.append(nn.Conv2d(
                in_channels if i == 0 else out_channels,
                out_channels, kernel_size=3, padding=1
            ))
            layers.append(nn.BatchNorm2d(out_channels))  # Modern addition
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.block(x)

class MiniVGG(nn.Module):
    """Simplified VGG for demonstration (smaller than VGG-16)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            VGGBlock(3, 64, 2),    # 32→16
            VGGBlock(64, 128, 2),  # 16→8
            VGGBlock(128, 256, 3), # 8→4
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

vgg = MiniVGG()
print(f"MiniVGG parameters: {sum(p.numel() for p in vgg.parameters()):,}")
x = torch.randn(1, 3, 32, 32)
print(f"Input: {x.shape} → Output: {vgg(x).shape}")

MiniVGG parameters: 1,740,362
Input: torch.Size([1, 3, 32, 32]) → Output: torch.Size([1, 10])


## 3. ResNet (2015) — The Game Changer 🏆

**The Problem:** Very deep networks suffer from vanishing gradients — adding more layers makes accuracy *worse*.

**The Solution:** **Skip connections** (residual connections) — the layer learns the *residual* F(x) and adds it to the input.

```
Standard:    x → [Conv-BN-ReLU-Conv-BN] → H(x)
Residual:    x → [Conv-BN-ReLU-Conv-BN] → F(x) + x → ReLU → H(x)
                   └──────────────────────────┘
                        Skip connection
```

**Why it works:** If the layer isn't useful, it can learn F(x) = 0, and the output is just x (identity). This makes it easy to train very deep networks (50, 101, 152 layers!).

**ADAS Impact:** ResNet is the **backbone** of most object detection models (Faster R-CNN, YOLO, etc.)

In [4]:
class ResidualBlock(nn.Module):
    """Basic residual block with skip connection."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        # Skip connection (if dimensions change)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        # Main path
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        # Skip connection: ADD input to output
        out += self.shortcut(x)  # ← This is the key innovation!
        out = self.relu(out)
        return out

# Test: verify the skip connection
block = ResidualBlock(64, 64)
x = torch.randn(1, 64, 16, 16)
print(f"ResidualBlock: {x.shape} → {block(x).shape}")

# With dimension change
block2 = ResidualBlock(64, 128, stride=2)
print(f"ResidualBlock (downsample): {x.shape} → {block2(x).shape}")

ResidualBlock: torch.Size([1, 64, 16, 16]) → torch.Size([1, 64, 16, 16])
ResidualBlock (downsample): torch.Size([1, 64, 16, 16]) → torch.Size([1, 128, 8, 8])


In [5]:
class ResNet18(nn.Module):
    """Simplified ResNet-18 implementation."""
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Initial convolution
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        
        # Residual layers
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        
        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = [ResidualBlock(in_channels, out_channels, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

resnet = ResNet18()
x = torch.randn(1, 3, 32, 32)
print(f"ResNet-18: {x.shape} → {resnet(x).shape}")
print(f"Parameters: {sum(p.numel() for p in resnet.parameters()):,}")

ResNet-18: torch.Size([1, 3, 32, 32]) → torch.Size([1, 10])
Parameters: 11,173,962


## 4. Using Pretrained Models (Transfer Learning)

In practice, you almost **never train from scratch**. Instead, use models pretrained on ImageNet (1.2M images, 1000 classes) and **fine-tune** for your task.

**For ADAS:** Pretrain on ImageNet → Fine-tune on driving datasets (nuScenes, KITTI, etc.)

In [6]:
# --- Using PyTorch's pretrained models ---

# Load pretrained ResNet-18 (uncomment to download ~44MB)
# pretrained_resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# For demonstration, load without pretrained weights
pretrained_resnet = models.resnet18(weights=None)

print("ResNet-18 architecture:")
print(f"Parameters: {sum(p.numel() for p in pretrained_resnet.parameters()):,}")

# Fine-tuning: replace the last FC layer
num_adas_classes = 5  # e.g., car, pedestrian, cyclist, truck, motorcycle
pretrained_resnet.fc = nn.Linear(512, num_adas_classes)
print(f"\nModified for ADAS ({num_adas_classes} classes):")
print(f"Output layer: {pretrained_resnet.fc}")

x = torch.randn(1, 3, 224, 224)
print(f"Input: {x.shape} → Output: {pretrained_resnet(x).shape}")

ResNet-18 architecture:


Parameters: 11,689,512



Modified for ADAS (5 classes):
Output layer: Linear(in_features=512, out_features=5, bias=True)
Input: torch.Size([1, 3, 224, 224]) → Output: torch.Size([1, 5])


In [7]:
# --- Architecture Comparison ---
architectures = {
    'LeNet-5': LeNet5(),
    'MiniVGG': MiniVGG(),
    'ResNet-18 (ours)': ResNet18(),
    'ResNet-18 (torchvision)': models.resnet18(weights=None),
    'ResNet-50': models.resnet50(weights=None),
    'MobileNetV2': models.mobilenet_v2(weights=None),
}

print(f"{'Architecture':<25} {'Parameters':>15} {'Notes'}")
print("-" * 70)
for name, model in architectures.items():
    params = sum(p.numel() for p in model.parameters())
    note = ""
    if 'MobileNet' in name:
        note = "← Efficient for edge/ADAS!"
    elif 'ResNet-50' in name:
        note = "← Common backbone in detection"
    print(f"{name:<25} {params:>12,}   {note}")

Architecture                   Parameters Notes
----------------------------------------------------------------------
LeNet-5                         61,706   
MiniVGG                      1,740,362   
ResNet-18 (ours)            11,173,962   
ResNet-18 (torchvision)     11,689,512   
ResNet-50                   25,557,032   ← Common backbone in detection
MobileNetV2                  3,504,872   ← Efficient for edge/ADAS!


---
## ✅ Key Takeaways

1. **LeNet → VGG → ResNet** shows the evolution from shallow to very deep networks
2. **Skip connections** (ResNet) solve vanishing gradients and enable 100+ layer networks
3. **Transfer learning** (pretrained models) is the standard approach — don't train from scratch!
4. **MobileNet** is designed for edge devices — critical for real-time ADAS
5. The **backbone** of most object detectors is a CNN (typically ResNet or similar)

---
## 📖 Next Steps
➡️ **Next notebook:** [03_object_detection_adas.ipynb](03_object_detection_adas.ipynb) — Object detection for ADAS